In [1]:
import xarray as xr
import pandas as pd
from scipy import stats
import numpy as np
from cartopy import crs as ccrs, feature as cfeature
from cartopy.mpl.ticker import LongitudeFormatter, LatitudeFormatter
import matplotlib.pyplot as plt
from metpy.units import units
import matplotlib.colors as colors
import metpy.calc as mpcalc
from metpy.calc import saturation_mixing_ratio
%run ../Feedback/Pendergrass-function.ipynb

ERROR 1: PROJ: proj_create_from_database: Open of /knight/anaconda_aug22/envs/aug22_env/share/proj failed


## Function for calculating the saturation specific humidity

In [2]:
def calcsatspechum(t,p):
    es = (1.0007+(3.46e-6*p))*6.1121*np.exp(17.502*(t-273.15)/(240.97+(t-273.15)))
    wsl = 0.622*es/(p-es) # saturation mixing ratio wrt liquid water (

    es = (1.0003+(4.18e-6*p))*6.1115*np.exp(22.452*(t-273.15)/(272.55+(t-273.15)))

    wsi = 0.622*es/(p-es) # saturation mixing ratio wrt ice 

    ws = wsl
#     ws(t<273.15)=wsi(t<273.15)
    ws=ws.where(t>=273.15,wsi)
    qs=ws/(1+ws) #% saturation specific humidity, g/kg
    return qs

In [3]:
def wv_feedback(Q1,T1,T2,dta,dq):
    #===================get the pressure and expand its dimension===================
    month=np.arange(1,13)
    lat=Q1.lat
    lon=Q1.lon
    p=Q1.lev_p
    press=p.expand_dims(dim={"month":month.size},axis=0)
    press=press.expand_dims(dim={"lat":lat.size},axis=2)
    press=press.expand_dims(dim={"lon":lon.size},axis=3)
    press.coords['month']=month
    press.coords['lat']=lat
    press.coords['lon']=lon
    #================preprocess all the dataset into the same coordinate============
    press=preprocess_3D(press)
    q1=preprocess_3D(Q1)
    t1=preprocess_3D(T1)
    t2=preprocess_3D(T2)
    dta=preprocess_3D(dta)
    #===============Calculate saturation q and dqdt==================================
    qs1 = calcsatspechum(t1,press)
    qs2 = calcsatspechum(t2,press)
    dqsdt = (qs2 - qs1)/dta
    rh = q1/qs1
    dqdt = rh*dqsdt
    #================Read the kernel================================================
    diri='/zhoulab_rit/lzhuo/data/Pendergrass_kernel/'
    filename='q.kernel.plev.nc'
    fi=xr.open_dataset(diri+filename)
    q_LW_kernel=fi.FLNT
    q_SW_kernel=fi.FSNT
    q_LW_kernel_cs=fi.FLNTC
    q_SW_kernel_cs=fi.FSNTC
    #===============preprocess all the dataset into the same coordinate before doing the calculation=====
    q_LW_kernel   =preprocess_3D(q_LW_kernel)
    q_SW_kernel   =preprocess_3D(q_SW_kernel)
    q_LW_kernel_cs=preprocess_3D(q_LW_kernel_cs)
    q_SW_kernel_cs=preprocess_3D(q_SW_kernel_cs)
    #==============Regrid dqdt onto the coordinate of the kernel=======================
    new_lat=q_LW_kernel.lat
    new_lon=q_LW_kernel.lon
    dqdt=dqdt.interp(lat=new_lat,lon=new_lon,kwargs={"fill_value": "extrapolate"})
    #===============Normalize the kernel================================================
    q_LW_kernel   =q_LW_kernel/dqdt
    q_SW_kernel   =q_SW_kernel/dqdt
    q_LW_kernel_cs=q_LW_kernel_cs/dqdt
    q_SW_kernel_cs=q_SW_kernel_cs/dqdt
    #=============preprocess dq===========================================================
    dQ=preprocess_3D(dq)
    #==========Regrid dq onto the coordinate of the kernel================================
    new_lat=q_LW_kernel.lat
    new_lon=q_LW_kernel.lon
    dQ=dQ.interp(lat=new_lat,lon=new_lon,kwargs={"fill_value": "extrapolate"})
    #============Do the calculation=======================================================
    LW_vert,SW_vert,dLW_q,dSW_q,total_q               =Water_vapor_feedback(q_LW_kernel,q_SW_kernel,dQ) #LW:+ upward; SW:+ downward; total:+ downward
    LW_vert_cs,SW_vert_cs,dLW_q_cs,dSW_q_cs,total_q_cs=Water_vapor_feedback(q_LW_kernel_cs,q_SW_kernel_cs,dQ)
    return dLW_q,dSW_q,total_q,dLW_q_cs,dSW_q_cs,total_q_cs